# Agent的高级用法

## 1、设置agent的名称

使用字段name来进行设置

举例：

In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 以init_chat_model为例
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)



agent = create_agent(
    model=model,
    name="agent01"
)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})
#
#
# rprint(response)

for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题
================================ Human Message =================================

100 + 20 * 3 = ？
================================== Ai Message ==================================
Name: agent01

按照运算顺序，先算乘法，再算加法：

- \(20 \times 3 = 60\)
- \(100 + 60 = 160\)

所以答案是：**160**


## 2、系统提示词的设置

使用system_prompt参数进行设置，可以是str,也可以是SystemMessage

举例1：

In [4]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)


# 1.导入模型
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.导入工具
web_search = TavilySearch(max_results=2)

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    name="agent02",
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='4ccc4da2-a5a6-43ed-8867-90d3bcf8f6b0'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 45,
                    'prompt_tokens': 1304,
                    'total_tokens': 1349,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 33,
                        'engine_ttlt_ms': 157,
                        'pre_inference_ms': 254,
                        'service_tbt_ms': 2,
                        'service_ttft_ms': 521,
                        'service_ttlt_ms': 611,
                        'user_visible_ttft_ms': 267
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Eg92ZS4TEr8426LS5QpcT3AvUop1G',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent02',
            id='lc_run--01a1a1a3-e0df-7440-900c-2bfd48bcddfc-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {
                        'query': '2026 FIFA World Cup host countries official',
                        'include_domains': ['fifa.com'],
                        'search_depth': 'basic',
                        'topic': 'general'
                    },
                    'id': 'call_sd56gNSWRilGlGRdsRy1Mxtc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1304,
                'output_tokens': 45,
                'total_tokens': 1349,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"query": "2026 FIFA World Cup host countries official", "follow_up_questions": null, 
"answer": null, "images": [], "results": [{"url": 
"https://fifaworldcup26.hospitality.fifa.com/blog/host-nation-deep-dive", "title": "Who Are The Host Nations of the
2026 FIFA World Cup?", "content": "FIFA logo\\nOnLocation logo\\nOnLocation logo\\nVISA & FIFA\\n\\n# Host Nation 
Deep Dive\\n\\nHere’s a look at the current football state of the three host countries’ national 
teams.\\n\\nShare\\n\\n.webp)\\n\\n## Three Nations, One Historic Stage\\n\\nFor the first time in history, the 
world’s greatest sporting event will be hosted by three nations. FIFA World Cup 2026™ will unite Canada, Mexico, 
and the United States in a tournament that promises scale, spectacle, and stories that will echo far beyond the 
final whistle. [...] For Canada, 2026 is a moment of arrival. A chance to win a FIFA World Cup™ match for the first
time ever. And, moreover, a chance to showcase a new football identity built on athleticism, tactical discipline, 
and fearless attacking play.\\n\\nHosting brings the opportunity to inspire nationwide growth while delivering 
competitive performances that announce the nation’s long-term presence on the global stage.\\n\\n### Mexico: 
Football Heritage on Home Soil [...] Under the leadership of experienced Ameri

举例2：

In [5]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"


# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)

rprint(response)
# print(response["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='10加上20再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='b8480bec-3673-40f1-bb94-764eb76c1c53'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 55,
                    'prompt_tokens': 158,
                    'total_tokens': 213,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 30,
                        'engine_ttlt_ms': 213,
                        'pre_inference_ms': 72,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 277,
                        'service_ttlt_ms': 447,
                        'user_visible_ttft_ms': 205
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Eg94PCiqRDDosFPDk4PlBUUakNtcv',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1a1a5-9df4-719b-8c65-16ab4dd0b719-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 10, 'b': 20},
                    'id': 'call_oVvXf3UTmYTFSRhm70fljEqf',
                    'type': 'tool_call'
                },
                {
                    'name': 'add_numbers',
                    'args': {'a': 30, 'b': 0},
                    'id': 'call_3cZnjAmUrpOOtd7CiwbfG7ub',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 158,
                'output_tokens': 55,
                'total_tokens': 213,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='af091bb5-bc4c-4e6c-b34f-4357d0955ff9',
            tool_call_id='call_oVvXf3UTmYTFSRhm70fljEqf'
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='044330bb-82e7-460f-8fe0-4bf23fa9a1e8',
            tool_call_id='call_3cZnjAmUrpOOtd7CiwbfG7ub'
        ),
        AIMessage(
            content='10 加上 20 再加上 30 的结果是 **60**。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 21,
                    'prompt_tokens': 238,
                    'total_tokens': 259,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 47,
                        'engine_ttlt_ms': 139,
                        'pre_inference_ms': 69,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 301,
                        

举例3：

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1

    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"


messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，杭州今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_cXQNHjH23oqlAsHeUEWzlHXc)
 Call ID: call_cXQNHjH23oqlAsHeUEWzlHXc
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Yv1BqXvriycxpkNBvzErM8Y3)
 Call ID: call_Yv1BqXvriycxpkNBvzErM8Y3
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_o1bB0Th8RFVbjBUxG3b2sPD8)
 Call ID: call_o1bB0Th8RFVbjBUxG3b2sPD8
  Args:
    city: 杭州
================================= To

## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略：

In [7]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ProviderStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='51361cf7-7c92-426e-9d74-ddd8cb6244b3'
        ),
        AIMessage(
            content='{"name":"小明","email":"shkstart@atguigu.com","phone":"13012341234"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 30,
                    'prompt_tokens': 242,
                    'total_tokens': 272,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 32,
                        'engine_ttlt_ms': 153,
                        'pre_inference_ms': 90,
                        'service_tbt_ms': 4,
                        'service_ttft_ms': 233,
                        'service_ttlt_ms': 351,
                        'user_visible_ttft_ms': 144
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Eg9ATb070x3GRxSWW8Vy66qBEI6UR',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a1a1ab-5c7c-73e7-b302-02622febeea4-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 242,
                'output_tokens': 30,
                'total_tokens': 272,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

### 3.2 ToolStrategy策略：

In [9]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ToolStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

# rprint(response)

print(response["structured_response"])

name='小明' email='shkstart@atguigu.com' phone='13012341234'


### 3.3 AutoStrategy / Type策略：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=AutoStrategy(ContractInfo)
    # response_format=ContractInfo  # 不推荐大家使用
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)

# print(response["structured_response"])